In [1]:
import requests
from requests.auth import HTTPBasicAuth

# El '0' indica "mi usuario actual"
url = "https://intervals.icu/api/v1/athlete/0/athlete-summary"

# Reemplaza con tu clave real obtenida en /settings
api_key = "1347lyp9p4meza7lyd3sar4qu" 

# La autenticación usa 'API_KEY' como usuario y tu clave como contraseña
auth = HTTPBasicAuth('API_KEY', api_key)

response = requests.get(url, auth=auth)

if response.status_code == 200:
    atletas = response.json()
    # Quitar duplicados por athlete_id y nombre
    atletas = list({(a.get('athlete_id'), a.get('athlete_name')): a for a in atletas}.values())
    print(f"Se encontraron {len(atletas)} atletas únicos.")
    print(atletas)
else:
    print(f"Error: {response.status_code}")

Se encontraron 3 atletas únicos.
[{'count': 9, 'time': 67914, 'moving_time': 67914, 'elapsed_time': 71714, 'calories': 18094, 'total_elevation_gain': 6026.0, 'training_load': 1145, 'srpe': 0, 'distance': 683057.56, 'eftp': None, 'eftpPerKg': None, 'date': '2026-01-26', 'athlete_id': 'i495562', 'athlete_name': 'CarlosGarcia11', 'email': None, 'external_id': None, 'fitness': 293.11163, 'fatigue': 229.84836, 'form': 63.263275, 'rampRate': -23.852325, 'weight': 62.0, 'timeInZones': [25435, 15404, 9984, 8747, 4860, 3100, 384, 8413], 'timeInZonesTot': 67914, 'byCategory': [{'count': 9, 'time': 67914, 'moving_time': 67914, 'elapsed_time': 71714, 'calories': 18094, 'total_elevation_gain': 6026.0, 'training_load': 1145, 'srpe': 0, 'distance': 683057.56, 'eftp': 361.18863, 'eftpPerKg': 5.8314543, 'category': 'Ride'}], 'mostRecentWellnessId': '2026-02-01'}, {'count': 2, 'time': 5901, 'moving_time': 5901, 'elapsed_time': 6228, 'calories': 1202, 'total_elevation_gain': 15.0, 'training_load': 118, '

In [ ]:
print("\n=== NOMBRES DE ATLETAS ===")
for atleta in atletas:
    print(f"- {atleta['athlete_name']}")


=== NOMBRES DE ATLETAS ===
- CarlosGarcia11
- Iosman
- david Echavarri


: 

In [ ]:
from datetime import datetime, timedelta
print("\n=== ESTADÍSTICAS DE ENTRENAMIENTO ===")
for i, atleta in enumerate(atletas, 1):
    nombre = atleta['athlete_name']
    fecha = atleta['date']
    distancia_m = atleta['distance']  # en metros
    distancia_km = distancia_m / 1000  # convertir a km
    tiempo_s = atleta['moving_time']  # en segundos
    calorias = atleta.get('calories', 0)
    kilojulios = calorias * 4.184  # 1 kcal = 4.184 kJ
    peso = atleta.get('weight')
    print(f"\n{i}. {nombre} - Fecha: {fecha}")
    # Convertir tiempo a hh:mm:ss
    horas = tiempo_s // 3600
    minutos = (tiempo_s % 3600) // 60
    segundos = tiempo_s % 60
    tiempo_formateado = f"{int(horas):02d}:{int(minutos):02d}:{int(segundos):02d}"
    
    # Calcular velocidad media en km/h
    tiempo_h = tiempo_s / 3600
    velocidad_media = distancia_km / tiempo_h if tiempo_h > 0 else 0
    
    # Calcular trabajo/peso (en kJ/kg)
    trabajo_por_peso = kilojulios / peso if peso and peso > 0 else 65
    print(nombre,fecha,distancia_km)
    # Configurar rango de fechas (últimos 90 días)
    oldest = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')
    newest = datetime.now().strftime('%Y-%m-%d')

    url_actividades = f"https://intervals.icu/api/v1/athlete/{atleta['athlete_id']}/activities?oldest={oldest}&newest={newest}"

    response_actividades = requests.get(url_actividades, auth=auth)

    if response_actividades.status_code == 200:
        actividades = response_actividades.json()
        print(f"\n=== ACTIVIDADES DEL ATLETA ===")
        print(f"Se encontraron {len(actividades)} actividades.\n")
    
        for i, actividad in enumerate(actividades, 1):
            
            fecha = actividad.get('start_date_local', 'N/A')
            nombre = actividad.get('name', 'Sin nombre')
            tipo_deporte = actividad.get('type', 'N/A')
            distancia = (actividad.get('distance') or 0) / 1000  # convertir a km
            tiempo = actividad.get('moving_time') or 0
        
            # Convertir tiempo a hh:mm:ss
            horas_a = tiempo // 3600
            minutos_a = (tiempo % 3600) // 60
            segundos_a = tiempo % 60
            tiempo_formateado_a = f"{int(horas_a):02d}:{int(minutos_a):02d}:{int(segundos_a):02d}"
        
            # Calcular velocidad o ritmo según el tipo de deporte
            if tipo_deporte == 'Run' and distancia > 0:
                # Para Run: calcular ritmo en min/km
                minutos_por_km = tiempo / 60 / distancia
                min_ritmo = int(minutos_por_km)
                seg_ritmo = int((minutos_por_km - min_ritmo) * 60)
                ritmo_str = f"{min_ritmo}:{seg_ritmo:02d} min/km"
                velocidad_str = ritmo_str
            else:
                # Para otros deportes: velocidad en km/h
                tiempo_h_a = tiempo / 3600
                velocidad = (distancia / tiempo_h_a) if tiempo_h_a > 0 else 0
                velocidad_str = f"{velocidad:.2f} km/h"
        
            print(f"{i}. {fecha} - {nombre} ({tipo_deporte})")
            print(f"   Distancia: {distancia:.2f} km | Tiempo: {tiempo_formateado_a} | Ritmo/Velocidad: {velocidad_str}")
        
            # Obtener detalles completos de la actividad para interval_summary
            id_actividad = actividad.get('id')
            url_detalle = f"https://intervals.icu/api/v1/athlete/{atleta['athlete_id']}/activities/{id_actividad}"
            response_detalle = requests.get(url_detalle, auth=auth)
            
            if response_detalle.status_code == 200:
                detalle_act = response_detalle.json()
                if isinstance(detalle_act, list) and len(detalle_act) > 0:
                    detalle_act = detalle_act[0]
                
                # Mostrar interval_summary si existe
                if 'interval_summary' in detalle_act and detalle_act['interval_summary']:
                    print(f"   Intervalos: {', '.join(detalle_act['interval_summary'])}")
            
                print()
        
            else:
                print(f"Error al obtener actividades: {response_actividades.status_code}")
                print(response_actividades.text)


=== ESTADÍSTICAS DE ENTRENAMIENTO ===

1. CarlosGarcia11 - Fecha: 2026-01-26
CarlosGarcia11 2026-01-26 683.0575600000001

=== ACTIVIDADES DEL ATLETA ===
Se encontraron 173 actividades.

1. 2026-01-31T12:59:18 - Andratx Ciclismo en ruta (Ride)
   Distancia: 41.28 km | Tiempo: 01:23:16 | Ritmo/Velocidad: 29.75 km/h
   Intervalos: 2x 74s 406w, 1x 52s 438w

2. 2026-01-30T12:16:50 - Selva Ciclismo en ruta (Ride)
   Distancia: 152.52 km | Tiempo: 03:42:03 | Ritmo/Velocidad: 41.21 km/h
   Intervalos: 6x 35s 428w, 12x 7s 605w, 1x 65s 430w, 1x 6m 390w, 2x 2m30s 369w, 3x 5m1s 346w, 1x 88s 377w, 1x 4m10s 393w, 1x 7m35s 337w

3. 2026-01-29T14:35:34 - Ses Salines Ciclismo en ruta (Ride)
   Distancia: 23.83 km | Tiempo: 00:26:51 | Ritmo/Velocidad: 53.26 km/h
   Intervalos: 5x 32s 448w, 1x 2m34s 396w

4. 2026-01-29T13:53:21 - Activación 3x3 (VirtualRide)
   Distancia: 0.00 km | Tiempo: 00:27:05 | Ritmo/Velocidad: 0.00 km/h
   Intervalos: 1x 5m36s 171w, 1x 10s 182w, 1x 8m2s 175w, 1x 5m38s 103w, 1x 61

In [ ]:
# Obtener las actividades del atleta


# Configurar rango de fechas (últimos 90 días)
oldest = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')
newest = datetime.now().strftime('%Y-%m-%d')

url_actividades = f"https://intervals.icu/api/v1/athlete/0/activities?oldest={oldest}&newest={newest}"

response_actividades = requests.get(url_actividades, auth=auth)

if response_actividades.status_code == 200:
    actividades = response_actividades.json()
    print(f"\n=== ACTIVIDADES DEL ATLETA ===")
    print(f"Se encontraron {len(actividades)} actividades.\n")
    
    for i, actividad in enumerate(actividades, 1):
        print("--------------------------------------------------"+str(actividad))
        fecha = actividad.get('start_date_local', 'N/A')
        nombre = actividad.get('name', 'Sin nombre')
        tipo_deporte = actividad.get('type', 'N/A')
        distancia = (actividad.get('distance') or 0) / 1000  # convertir a km
        tiempo = actividad.get('moving_time') or 0
        
        # Convertir tiempo a hh:mm:ss
        horas_a = tiempo // 3600
        minutos_a = (tiempo % 3600) // 60
        segundos_a = tiempo % 60
        tiempo_formateado_a = f"{int(horas_a):02d}:{int(minutos_a):02d}:{int(segundos_a):02d}"
        
        # Calcular velocidad o ritmo según el tipo de deporte
        if tipo_deporte == 'Run' and distancia > 0:
            # Para Run: calcular ritmo en min/km
            minutos_por_km = tiempo / 60 / distancia
            min_ritmo = int(minutos_por_km)
            seg_ritmo = int((minutos_por_km - min_ritmo) * 60)
            ritmo_str = f"{min_ritmo}:{seg_ritmo:02d} min/km"
            velocidad_str = ritmo_str
        else:
            # Para otros deportes: velocidad en km/h
            tiempo_h_a = tiempo / 3600
            velocidad = (distancia / tiempo_h_a) if tiempo_h_a > 0 else 0
            velocidad_str = f"{velocidad:.2f} km/h"
        
        print(f"{i}. {fecha} - {nombre} ({tipo_deporte})")
        print(f"   Distancia: {distancia:.2f} km | Tiempo: {tiempo_formateado_a} | Ritmo/Velocidad: {velocidad_str}")
        
        # Obtener detalles completos de la actividad para interval_summary
        id_actividad = actividad.get('id')
        url_detalle = f"https://intervals.icu/api/v1/athlete/{id}/activities/{id_actividad}"
        response_detalle = requests.get(url_detalle, auth=auth)
        
        if response_detalle.status_code == 200:
            detalle_act = response_detalle.json()
            if isinstance(detalle_act, list) and len(detalle_act) > 0:
                detalle_act = detalle_act[0]
            
            # Mostrar interval_summary si existe
            if 'interval_summary' in detalle_act and detalle_act['interval_summary']:
                print(f"   Intervalos: {', '.join(detalle_act['interval_summary'])}")
        
        print()
        
else:
    print(f"Error al obtener actividades: {response_actividades.status_code}")
    print(response_actividades.text)


=== ACTIVIDADES DEL ATLETA ===
Se encontraron 78 actividades.

--------------------------------------------------{'id': 'i121221690', 'start_date_local': '2026-01-28T18:54:05', 'type': 'Run', 'icu_ignore_time': False, 'icu_pm_cp': 364, 'icu_pm_w_prime': 31263, 'icu_pm_p_max': 650, 'icu_pm_ftp': 372, 'icu_pm_ftp_secs': 720, 'icu_pm_ftp_watts': 408, 'icu_ignore_power': False, 'icu_rolling_cp': None, 'icu_rolling_w_prime': 33046.484, 'icu_rolling_p_max': 908.9413, 'icu_rolling_ftp': 390, 'icu_rolling_ftp_delta': 0, 'icu_training_load': 84, 'icu_atl': 94.50471, 'icu_ctl': 91.00332, 'ss_p_max': 36.14613, 'ss_w_prime': 257.93475, 'ss_cp': 1057.7894, 'paired_event_id': None, 'icu_ftp': 300, 'icu_joules': 700040, 'icu_recording_time': 1822, 'elapsed_time': 1922, 'icu_weighted_avg_watts': 387, 'carbs_used': None, 'name': 'Challenge 1,como los niños de charco en charco jajajaj', 'description': '🦶⬆️ 2026 = 759 m | 🌐 summitbag.com', 'start_date': '2026-01-28T17:54:05Z', 'distance': 6190.78, 'icu_

In [ ]:
# Obtener mejores potencias (5', 10', 20') de actividades
print("\n=== MEJORES ESFUERZOS DE POTENCIA ===\n")

# Primero, contar cuántas actividades tienen datos de potencia
actividades_con_potencia = [a for a in actividades if a.get('avg_watts') and a.get('avg_watts') > 0]
print(f"Actividades con datos de potencia: {len(actividades_con_potencia)} de {len(actividades)}\n")

if not actividades_con_potencia:
    print("No se encontraron actividades con datos de potencia.")
else:
    # Intervalos de tiempo en segundos
    intervalos = [300, 600, 1200]  # 5min, 10min, 20min
    
    for i, actividad in enumerate(actividades_con_potencia[:5], 1):  # Mostrar las primeras 5
        id_actividad = actividad.get('id')
        fecha = actividad.get('start_date_local', 'N/A')
        nombre = actividad.get('name', 'Sin nombre')
        tipo_deporte = actividad.get('type', 'N/A')
        avg_watts = actividad.get('avg_watts')
        
        # Obtener detalles completos de la actividad
        url_detalle = f"https://intervals.icu/api/v1/athlete/0/activities/{id_actividad}"
        response_detalle = requests.get(url_detalle, auth=auth)
        print(f"Actividad ID: {id_actividad}")
        print(response_detalle.text)
        if response_detalle.status_code == 200:
            detalle = response_detalle.json()
            
            print(f"{i}. {fecha} - {nombre} ({tipo_deporte})")
            print(f"   Potencia media: {avg_watts:.0f}W")
            
            # Buscar los mejores esfuerzos en icu_intervals
            if 'icu_intervals' in detalle and detalle['icu_intervals']:
                potencias = {}
                for interval in detalle['icu_intervals']:
                    secs = interval.get('secs')
                    watts = interval.get('watts')
                    if secs in intervalos and watts:
                        tiempo_min = secs // 60
                        potencias[f"{tiempo_min}min"] = watts
                
                if potencias:
                    potencias_str = ' | '.join([f"{k}: {v:.0f}W" for k, v in sorted(potencias.items())])
                    print(f"   Mejores esfuerzos: {potencias_str}")
                else:
                    print("   Mejores esfuerzos: No disponibles en icu_intervals")
            else:
                print("   Mejores esfuerzos: No disponibles")
            
            print()
        else:
            print(f"   Error al obtener detalles: {response_detalle.status_code}\n")


=== MEJORES ESFUERZOS DE POTENCIA ===

Actividades con datos de potencia: 0 de 78

No se encontraron actividades con datos de potencia.


In [ ]:
# Buscar mejores 10 minutos de potencia en response_detalle
if response_detalle.status_code == 200:
    detalle = response_detalle.json()
    
    print("=== BÚSQUEDA DE MEJORES 10 MINUTOS DE POTENCIA ===\n")
    print(f"Tipo de respuesta: {type(detalle)}")
    
    # Si es una lista, trabajar con el primer elemento
    if isinstance(detalle, list):
        print(f"La respuesta es una lista con {len(detalle)} elementos")
        if len(detalle) > 0:
            detalle = detalle[0]
            print(f"Trabajando con el primer elemento\n")
        else:
            print("La lista está vacía")
            detalle = None
    
    if detalle:
        # Buscar en icu_intervals
        if 'icu_intervals' in detalle and detalle['icu_intervals']:
            print(f"Total de intervalos en icu_intervals: {len(detalle['icu_intervals'])}\n")
            
            # Buscar específicamente 600 segundos (10 minutos)
            for interval in detalle['icu_intervals']:
                secs = interval.get('secs')
                if secs == 600:  # 10 minutos
                    watts = interval.get('watts')
                    print(f"✓ Encontrado: 10 minutos (600 seg)")
                    print(f"  Potencia: {watts}W")
                    print(f"  Datos completos del intervalo: {interval}")
                    break
            else:
                print("✗ No se encontró intervalo de 10 minutos (600 seg)")
                print("\nIntervalos disponibles (primeros 20):")
                for interval in detalle['icu_intervals'][:20]:
                    secs = interval.get('secs')
                    watts = interval.get('watts')
                    print(f"  {secs}seg ({secs//60}min {secs%60}seg): {watts}W")
        else:
            print("No hay campo 'icu_intervals' en la respuesta")
            print("\nCampos disponibles en la respuesta:")
            if isinstance(detalle, dict):
                for campo in sorted(detalle.keys()):
                    if 'power' in campo.lower() or 'watts' in campo.lower() or 'interval' in campo.lower():
                        print(f"  - {campo}: {detalle[campo]}")
            else:
                print(f"Estructura inesperada: {type(detalle)}")
                print(detalle)
else:
    print(f"Error en response_detalle: {response_detalle.status_code}")
    print(response_detalle.text)

Error en response_detalle: 403
{"status":403,"error":"Access denied"}


In [ ]:
# Analizar intervalos de potencia de la actividad
if response_detalle.status_code == 200:
    detalle = response_detalle.json()
    if isinstance(detalle, list) and len(detalle) > 0:
        detalle = detalle[0]
    
    print("=== ANÁLISIS DE INTERVALOS DE POTENCIA ===\n")
    
    # Información general de potencia
    if 'icu_average_watts' in detalle:
        print(f"Potencia media: {detalle['icu_average_watts']}W")
    if 'icu_weighted_avg_watts' in detalle:
        print(f"Potencia normalizada: {detalle['icu_weighted_avg_watts']}W")
    
    # Analizar interval_summary
    if 'interval_summary' in detalle and detalle['interval_summary']:
        print(f"\nResumen de intervalos encontrados:")
        for intervalo in detalle['interval_summary']:
            print(f"  • {intervalo}")
            
            # Buscar el intervalo cercano a 10 minutos
            if '10m' in intervalo:
                print(f"    ⭐ Este es cercano a 10 minutos!")
    
    print("\n" + "="*50)
    print("MEJOR ESFUERZO DE ~10 MINUTOS:")
    # Extraer el intervalo de ~10 minutos
    for intervalo in detalle.get('interval_summary', []):
        if '10m' in intervalo:
            print(f"  {intervalo}")
            # Intentar extraer la potencia
            import re
            match = re.search(r'(\d+)w', intervalo)
            if match:
                potencia_10min = match.group(1)
                print(f"  Potencia: {potencia_10min}W")
    print("="*50)
else:
    print(f"Error: {response_detalle.status_code}")

Error: 403
